In [ ]:
import numpy as np
import tensorflow as tf

In [ ]:
IdX = np.load("XId_AllSteps.npy", allow_pickle=True).item()
# X = np.memmap("X_AllSteps_f32_norm.memmap", dtype=np.float32, mode="r", shape=(183199, 101, 75, 40))
# X = np.memmap("X_AllStepsR_f32_norm.memmap", dtype=np.float32, mode="r", shape=(183199, 101, 75, 40))
X = np.memmap("X_AllStepsR_SmoothA_f32_norm.memmap", dtype=np.float32, mode="r", shape=(183199, 101, 75, 40))
Y = np.load("Y_AllSteps.npy")       # class 0 -> 150

In [ ]:
X.shape, len(IdX)

In [ ]:
SIDE_LEFT = 1
SIDE_RIGHT = 0

In [ ]:
class SupervisedContrastiveLoss(tf.keras.losses.Loss):
    def __init__(self, temperature=0.07, name="supcon"):
        super().__init__(name=name)
        self.temperature = temperature

    def call(self, labels, features):
        labels = tf.reshape(labels, [-1])
        labels = tf.cast(labels, tf.int32)

        features = tf.math.l2_normalize(features, axis=1)

        logits = tf.matmul(features, features, transpose_b=True)
        logits = logits / self.temperature

        mask = tf.equal(
            tf.expand_dims(labels, 1),
            tf.expand_dims(labels, 0)
        )

        logits_mask = tf.ones_like(mask, dtype=tf.float32) - tf.eye(tf.shape(labels)[0])
        mask = tf.cast(mask, tf.float32) * logits_mask

        exp_logits = tf.exp(logits) * logits_mask
        log_prob = logits - tf.math.log(tf.reduce_sum(exp_logits, axis=1, keepdims=True) + 1e-9)

        mean_log_prob_pos = tf.reduce_sum(mask * log_prob, axis=1) / (
            tf.reduce_sum(mask, axis=1) + 1e-9
        )

        loss = -tf.reduce_mean(mean_log_prob_pos)
        return loss

### V9. 1D architecture with both foot, concat on series length, contrastive loss, custom-made batches

In [ ]:
def batch_generator_contrastive_concat_sides_series(X, Y, IdX, flat=True):
    pids = np.array(list(IdX.keys()))
    shoes = np.array(list(IdX[1].keys()))
    speeds = np.array(list(IdX[1][0].keys()))

    while True:
        X_ = np.zeros((32, 101*2, 75, 40), dtype=np.float32)
        Y_ = np.zeros((32, 1), dtype=np.float32)

        pids_ = [np.random.choice(pids), np.random.choice(pids), np.random.choice(pids), np.random.choice(pids)]
        shoes_ = [np.random.choice(shoes), np.random.choice(shoes), np.random.choice(shoes), np.random.choice(shoes)]
        speeds_ = [np.random.choice(speeds), np.random.choice(speeds), np.random.choice(speeds), np.random.choice(speeds)]

        i = 0
        for curr_setup in range(4):
            for curr_pid in pids_:
                T = IdX[curr_pid][shoes_[curr_setup]][speeds_[curr_setup]]
                RL = np.random.randint(0, len(T[SIDE_LEFT]))
                RR = np.random.randint(0, len(T[SIDE_RIGHT]))
                X_[i, 0:101] = X[T[SIDE_LEFT][RL]]
                X_[i, 101:] = X[T[SIDE_RIGHT][RR]]
                # assert Y[T[SIDE_LEFT][RL], 0:1] == Y[T[SIDE_RIGHT][RR], 0:1]
                Y_[i] = Y[T[SIDE_LEFT][RL], 0:1]
                i += 1

        for _ in range(4):
            for curr_pid in pids_:
                shoe = np.random.choice(shoes)
                speed = np.random.choice(speeds)
                T = IdX[curr_pid][shoe][speed]
                RL = np.random.randint(0, len(T[SIDE_LEFT]))
                RR = np.random.randint(0, len(T[SIDE_RIGHT]))
                X_[i, 0:101] = X[T[SIDE_LEFT][RL]]
                X_[i, 101:] = X[T[SIDE_RIGHT][RR]]
                # assert Y[T[SIDE_LEFT][RL], 0:1] == Y[T[SIDE_RIGHT][RR], 0:1]
                Y_[i] = Y[T[SIDE_LEFT][RL], 0:1]
                i += 1
        
        if flat:
            X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
            
        # yield X_, Y_
        yield X_, {"dense": Y_, "lambda": Y_}

gen_train = batch_generator_contrastive_concat_sides_series(X, Y, IdX)

In [ ]:
A, B = next(gen_train)
A[0].shape

In [ ]:
import inceptionembed
model = inceptionembed.get_model(n_classes=200, input_shape=(None, 75*40), reduce=[tf.keras.layers.GlobalAveragePooling1D()])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),

    loss={
        "dense": tf.keras.losses.SparseCategoricalCrossentropy(),
        "lambda": SupervisedContrastiveLoss()
    },

    loss_weights={
        "dense": 1.0,
        "lambda": 0.2
    },

    metrics={
        "dense": tf.keras.metrics.SparseCategoricalAccuracy(name="acc")
    }
)

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

class Every50Epochs(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 50 == 0:
            self.model.save(f'checkpoint_epoch_{epoch+1}.keras')
            print(f"\nSaved checkpoint at epoch {epoch+1}")

In [ ]:
history = model.fit(
    gen_train,
    steps_per_epoch=len(Y)//32//2, # Real epoch as there is two times side each time
    epochs=300,
    callbacks=[checkpoint, Every50Epochs()]
)

In [ ]:
model.save('last_model.keras')

In [ ]:
model.load_weights('best_model.keras')
model.evaluate(gen_train, steps=len(Y)//32)

In [ ]:
model.load_weights('last_model.keras')
model.evaluate(gen_train, steps=len(Y)//32)

### V8. 1D architecture with both foot, concat on series channel, contrastive loss, custom-made batches

In [ ]:
def batch_generator_contrastive_concat_sides_channel(X, Y, IdX, flat=True):
    pids = np.array(list(IdX.keys()))
    shoes = np.array(list(IdX[1].keys()))
    speeds = np.array(list(IdX[1][0].keys()))

    while True:
        X_ = np.zeros((32, 101, 75, 40, 2), dtype=np.float32)
        Y_ = np.zeros((32, 1), dtype=np.float32)

        pids_ = [np.random.choice(pids), np.random.choice(pids), np.random.choice(pids), np.random.choice(pids)]
        shoes_ = [np.random.choice(shoes), np.random.choice(shoes), np.random.choice(shoes), np.random.choice(shoes)]
        speeds_ = [np.random.choice(speeds), np.random.choice(speeds), np.random.choice(speeds), np.random.choice(speeds)]

        i = 0
        for curr_setup in range(4):
            for curr_pid in pids_:
                T = IdX[curr_pid][shoes_[curr_setup]][speeds_[curr_setup]]
                RL = np.random.randint(0, len(T[SIDE_LEFT]))
                RR = np.random.randint(0, len(T[SIDE_RIGHT]))
                X_[i, :, :, :, 0] = X[T[SIDE_LEFT][RL]]
                X_[i, :, :, :, 1] = X[T[SIDE_RIGHT][RR]]
                # assert Y[T[SIDE_LEFT][RL], 0:1] == Y[T[SIDE_RIGHT][RR], 0:1]
                Y_[i] = Y[T[SIDE_LEFT][RL], 0:1]
                i += 1

        for _ in range(4):
            for curr_pid in pids_:
                shoe = np.random.choice(shoes)
                speed = np.random.choice(speeds)
                T = IdX[curr_pid][shoe][speed]
                RL = np.random.randint(0, len(T[SIDE_LEFT]))
                RR = np.random.randint(0, len(T[SIDE_RIGHT]))
                X_[i, :, :, :, 0] = X[T[SIDE_LEFT][RL]]
                X_[i, :, :, :, 1] = X[T[SIDE_RIGHT][RR]]
                # assert Y[T[SIDE_LEFT][RL], 0:1] == Y[T[SIDE_RIGHT][RR], 0:1]
                Y_[i] = Y[T[SIDE_LEFT][RL], 0:1]
                i += 1
        
        if flat:
            X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
            
        # yield X_, Y_
        yield X_, {"dense": Y_, "lambda": Y_}

gen_train = batch_generator_contrastive_concat_sides_channel(X, Y, IdX)

In [ ]:
A, B = next(gen_train)
A[0].shape

In [ ]:
import inceptionembed
model = inceptionembed.get_model(n_classes=200, input_shape=(None, 75*40*2), reduce=[tf.keras.layers.GlobalAveragePooling1D()])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),

    loss={
        "dense": tf.keras.losses.SparseCategoricalCrossentropy(),
        "lambda": SupervisedContrastiveLoss()
    },

    loss_weights={
        "dense": 1.0,
        "lambda": 0.2
    },

    metrics={
        "dense": tf.keras.metrics.SparseCategoricalAccuracy(name="acc")
    }
)

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model.keras',
    monitor='loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

class Every25Epochs(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 25 == 0:
            self.model.save(f'checkpoint_epoch_{epoch+1}.keras')
            print(f"\nSaved checkpoint at epoch {epoch+1}")

In [ ]:
history = model.fit(
    gen_train,
    steps_per_epoch=len(Y)//32//2, # Real epoch as there is two times side each time
    epochs=300,
    callbacks=[checkpoint, Every25Epochs()]
)

In [ ]:
model.save('last_model.keras')

In [ ]:
model.load_weights('best_model.keras')
model.evaluate(gen_train, steps=len(Y)//32)

In [ ]:
model.load_weights('last_model.keras')
model.evaluate(gen_train, steps=len(Y)//32)